# Week 5 Exercise – Beat the Numbers

Improved RAG pipeline for the Insurellm evaluation: **query rewriting**, **retrieve 20 → LLM rerank → top 10**, and a **stricter answer prompt** (accuracy, completeness, relevance).

Run this notebook from the **week5** directory so `evaluation` and `implementation` resolve (e.g. in terminal: `cd week5 && jupyter notebook` or run cells from Cursor with week5 as cwd).

In [9]:
import sys
from pathlib import Path

cwd = Path.cwd()
if (cwd / "evaluation").exists():
    week5 = cwd
elif (cwd / "week5" / "evaluation").exists():
    week5 = cwd / "week5"
else:
    week5 = cwd.parent.parent.parent
sys.path.insert(0, str(week5))

import improved_rag
import evaluation.eval as eval_mod
from evaluation.test import load_tests

eval_mod.fetch_context = improved_rag.fetch_context
eval_mod.answer_question = improved_rag.answer_question

tests = load_tests()
print(f"Loaded {len(tests)} tests")

Loaded 150 tests


## Single-test check (retrieval + answer)

In [10]:
test = tests[0]
print("Question:", test.question)
print("Category:", test.category)
print("Keywords:", test.keywords)

r = eval_mod.evaluate_retrieval(test)
print(f"\nRetrieval: MRR={r.mrr:.4f}, nDCG={r.ndcg:.4f}, coverage={r.keyword_coverage:.1f}%")

ans_eval, answer, docs = eval_mod.evaluate_answer(test)
print(f"\nAnswer: {answer[:200]}...")
print(f"Scores: accuracy={ans_eval.accuracy:.2f}, completeness={ans_eval.completeness:.2f}, relevance={ans_eval.relevance:.2f}")

Question: Who won the prestigious IIOTY award in 2023?
Category: direct_fact
Keywords: ['Maxine', 'Thompson', 'IIOTY']

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



AuthenticationError: litellm.AuthenticationError: AuthenticationError: OpenAIException - Incorrect API key provided: sk-or-v1*************************************************************639a. You can find your API key at https://platform.openai.com/account/api-keys.

## Full evaluation

Run the full retrieval and answer evaluation (150 tests each). This will take a while due to LLM calls for rewrite, rerank, answer, and judge.

In [ ]:
from collections import defaultdict

print("Retrieval evaluation...")
total_mrr, total_ndcg, total_cov, n = 0.0, 0.0, 0.0, 0
for test, result, _ in eval_mod.evaluate_all_retrieval():
    n += 1
    total_mrr += result.mrr
    total_ndcg += result.ndcg
    total_cov += result.keyword_coverage
avg_mrr = total_mrr / n
avg_ndcg = total_ndcg / n
avg_cov = total_cov / n
print(f"  MRR: {avg_mrr:.4f}, nDCG: {avg_ndcg:.4f}, Keyword coverage: {avg_cov:.1f}%")

print("\nAnswer evaluation...")
total_acc, total_comp, total_rel, n = 0.0, 0.0, 0.0, 0
for test, result, _ in eval_mod.evaluate_all_answers():
    n += 1
    total_acc += result.accuracy
    total_comp += result.completeness
    total_rel += result.relevance
print(f"  Accuracy: {total_acc/n:.2f}/5, Completeness: {total_comp/n:.2f}/5, Relevance: {total_rel/n:.2f}/5")